## Iteration 9 — End-to-End RAG Evaluation

### Final retrieval pipeline

FAISS Top-10 → MMR Top-5 → Cross-Encoder reranking → Grounded answer generation

this notebook does not invent expected answers or calculate a misleading answer-accuracy score.
Instead it evaluates:

#### Retrieval

- Any expected chunk — Top-1
- Any expected chunk — Top-3
- Any expected chunk — Top-5
- All expected chunks — Top-1
- All expected chunks — Top-3
- All expected chunks — Top-5

#### Generation

- Generated answer
- Retrieved evidence
- Source/chunk citations requested in the answer
- Manual correctness/completeness/grounding/hallucination review

If you later add an expected_answer field to test_prompts.json, the notebook can also calculate semantic similarity.


In [40]:
# Install required packages
# Avoid reinstalling PyTorch unnecessarily on an existing local Python environment.
!pip -q install -U sentence-transformers faiss-cpu pandas numpy transformers sentencepiece


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
import os
import json
import re
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IPython.display import display

print("Python:", __import__("sys").version)
print("PyTorch:", torch.__version__)

try:
    import transformers
    print("Transformers:", transformers.__version__)
except Exception as e:
    print("Transformers import/version issue:", e)

print("Libraries loaded.")

Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.14.1+cpu
Transformers: 5.18.0
Libraries loaded.


### 1. Load the RMIT chunks and test questions

Change the two paths only if your files are stored elsewhere.


In [43]:
CHUNKS_PATH = "./02_rmit_assessment_support_chunks.csv"
TEST_PATH = "./test_prompts.json"

if not os.path.exists(CHUNKS_PATH):
    raise FileNotFoundError(
        f"Could not find {CHUNKS_PATH}. "
        "Put the CSV in the same folder as this notebook."
    )

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(
        f"Could not find {TEST_PATH}. "
        "Put test_prompts.json in the same folder as this notebook."
    )

chunks_df = pd.read_csv(CHUNKS_PATH)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

if "tests" not in test_data:
    raise ValueError(
        "Expected test_prompts.json to contain a top-level 'tests' list."
    )

test_df = pd.DataFrame(test_data["tests"])

required_test_columns = {"prompt", "chunks"}

missing = required_test_columns - set(test_df.columns)

if missing:
    raise ValueError(
        f"Missing required test columns: {missing}"
    )

print("Chunks:", len(chunks_df))
print("Test questions:", len(test_df))
print("Test columns:", test_df.columns.tolist())

display(test_df[["prompt", "chunks"]])

Chunks: 71
Test questions: 12
Test columns: ['prompt', 'chunks']


,prompt,chunks
0,I've got a timed quiz that I won't be able to ...,[EXT-01_1]
1,"I've recently lost my job, is this a valid rea...",[SC-01_7]
2,So my hard drive with my assignment stopped wo...,[SC-01_16]
3,I've just booked my first appointment with ELS...,[EAA-02_29]
4,I emailed my course coordinator three days ago...,[EAA-02_38]
5,Having a hard time reading through journal art...,[EAA-02_46]
6,My mate got flagged for plagiarism because he ...,[POL-01_70]
7,Who actually sets RMIT's official academic cal...,[POL-01_67]
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]"
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]"


### 2. Check whether gold answers are available

The current test JSON only has expected chunk IDs.

This is important: retrieval evaluation and answer evaluation are different tasks.

Without a gold answer, we can generate and manually review the RAG answer, but we should not claim an automatic answer-accuracy percentage.


In [44]:
ANSWER_FIELD = None

possible_answer_fields = [
    "expected_answer",
    "answer",
    "reference_answer",
    "gold_answer",
    "ground_truth",
    "ground_truth_answer"
]

for field in possible_answer_fields:
    if field in test_df.columns:
        ANSWER_FIELD = field
        break

if ANSWER_FIELD:
    print("Gold answer field found:", ANSWER_FIELD)
else:
    print("No gold/expected answer field found.")
    print("Automatic answer-accuracy scoring will be skipped.")

No gold/expected answer field found.
Automatic answer-accuracy scoring will be skipped.


#### Part A — Iteration 8 Retrieval

3. Build the same document representation used in the winning experiment

category + section + content


In [45]:
chunks_df["embedding_text"] = (
    chunks_df["category"].fillna("") + ". " +
    chunks_df["section"].fillna("") + ". " +
    chunks_df["content"].fillna("")
)

print(chunks_df.loc[0, "embedding_text"][:500])

Extension. If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date.. If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seven days , you must apply for special consid


In [46]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(embeddings)

faiss_index = faiss.IndexFlatIP(
    embeddings.shape[1]
)

faiss_index.add(embeddings)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("FAISS vectors:", faiss_index.ntotal)
print("Embedding dimension:", embeddings.shape[1])

Batches: 100%|██████████| 3/3 [00:00<00:00,  3.23it/s]

Embedding model: all-MiniLM-L6-v2
FAISS vectors: 71
Embedding dimension: 384


In [47]:
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(
    RERANKER_MODEL_NAME
)

print("Cross-encoder:", RERANKER_MODEL_NAME)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2879.20it/s]


Cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


In [48]:
def faiss_candidates(question, k=10):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(
        query_embedding,
        k
    )

    return query_embedding[0], [
        {
            "index": int(idx),
            "faiss_rank": rank + 1,
            "faiss_score": float(score)
        }
        for rank, (idx, score)
        in enumerate(zip(indices[0], scores[0]))
    ]


def mmr_select(
    question,
    candidate_k=10,
    final_k=5,
    lambda_param=0.70
):

    query_embedding, candidates = faiss_candidates(
        question,
        candidate_k
    )

    candidate_indices = [
        x["index"] for x in candidates
    ]

    candidate_embeddings = embeddings[
        candidate_indices
    ]

    relevance_scores = np.dot(
        candidate_embeddings,
        query_embedding.reshape(-1, 1)
    ).flatten()

    selected_positions = []
    remaining_positions = list(
        range(len(candidate_indices))
    )

    while remaining_positions and len(selected_positions) < final_k:

        best_position = None
        best_score = -np.inf

        for position in remaining_positions:

            if selected_positions:
                similarities = np.dot(
                    candidate_embeddings[selected_positions],
                    candidate_embeddings[position]
                )

                redundancy = float(
                    np.max(similarities)
                )
            else:
                redundancy = 0.0

            mmr_score = (
                lambda_param * relevance_scores[position]
                -
                (1 - lambda_param) * redundancy
            )

            if mmr_score > best_score:
                best_score = mmr_score
                best_position = position

        selected_positions.append(best_position)
        remaining_positions.remove(best_position)

    rows = []

    for mmr_rank, position in enumerate(
        selected_positions,
        start=1
    ):

        idx = candidate_indices[position]
        row = chunks_df.iloc[idx]
        candidate = candidates[position]

        rows.append({
            "mmr_rank": mmr_rank,
            "index": idx,
            "chunk_id": row["chunk_id"],
            "source_id": row["source_id"],
            "category": row["category"],
            "section": row["section"],
            "content": row["content"],
            "embedding_text": row["embedding_text"],
            "faiss_rank": candidate["faiss_rank"],
            "faiss_score": candidate["faiss_score"]
        })

    return pd.DataFrame(rows)


def mmr_then_rerank(
    question,
    candidate_k=10,
    mmr_k=5,
    lambda_param=0.70
):

    candidates = mmr_select(
        question,
        candidate_k,
        mmr_k,
        lambda_param
    ).copy()

    pairs = [
        (question, row["embedding_text"])
        for _, row in candidates.iterrows()
    ]

    candidates["reranker_score"] = (
        reranker.predict(pairs)
    )

    candidates = (
        candidates
        .sort_values(
            "reranker_score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    candidates["final_rank"] = (
        np.arange(len(candidates)) + 1
    )

    return candidates

#### 4. Run retrieval evaluation

This should reproduce the Iteration 8 retrieval results as closely as possible.


In [49]:
def any_expected(retrieved, expected):
    return any(
        chunk in retrieved
        for chunk in expected
    )


def all_expected(retrieved, expected):
    return all(
        chunk in retrieved
        for chunk in expected
    )


retrieval_rows = []

for test_id, row in test_df.iterrows():

    question = str(row["prompt"])
    expected = row["chunks"]

    retrieved_df = mmr_then_rerank(
        question,
        candidate_k=10,
        mmr_k=5,
        lambda_param=0.70
    )

    retrieved = retrieved_df["chunk_id"].tolist()

    ranks = [
        retrieved.index(chunk) + 1
        for chunk in expected
        if chunk in retrieved
    ]

    retrieval_rows.append({
        "test_id": test_id,
        "prompt": question,
        "expected_chunks": expected,
        "top1_chunk": retrieved[:1],
        "top3_chunks": retrieved[:3],
        "top5_chunks": retrieved[:5],
        "expected_rank_in_top5": (
            min(ranks) if ranks else None
        ),
        "any_top1": any_expected(
            retrieved[:1],
            expected
        ),
        "any_top3": any_expected(
            retrieved[:3],
            expected
        ),
        "any_top5": any_expected(
            retrieved[:5],
            expected
        ),
        "all_top1": all_expected(
            retrieved[:1],
            expected
        ),
        "all_top3": all_expected(
            retrieved[:3],
            expected
        ),
        "all_top5": all_expected(
            retrieved[:5],
            expected
        )
    })

retrieval_df = pd.DataFrame(
    retrieval_rows
)

retrieval_summary = pd.DataFrame({
    "Metric": [
        "Any expected — Top-1",
        "Any expected — Top-3",
        "Any expected — Top-5",
        "All expected — Top-1",
        "All expected — Top-3",
        "All expected — Top-5"
    ],
    "Result": [
        retrieval_df["any_top1"].mean(),
        retrieval_df["any_top3"].mean(),
        retrieval_df["any_top5"].mean(),
        retrieval_df["all_top1"].mean(),
        retrieval_df["all_top3"].mean(),
        retrieval_df["all_top5"].mean()
    ]
})

display(
    retrieval_summary.style.format(
        {"Result": "{:.1%}"}
    )
)

,Metric,Result
0,Any expected — Top-1,66.7%
1,Any expected — Top-3,83.3%
2,Any expected — Top-5,100.0%
3,All expected — Top-1,58.3%
4,All expected — Top-3,75.0%
5,All expected — Top-5,83.3%


In [50]:
top5_failures = retrieval_df[
    retrieval_df["any_top5"] == False
]

print(
    "Iteration 8 Top-5 failures:",
    len(top5_failures)
)

display(
    top5_failures[
        [
            "test_id",
            "prompt",
            "expected_chunks",
            "top5_chunks",
            "expected_rank_in_top5"
        ]
    ]
)

Iteration 8 Top-5 failures: 0


,test_id,prompt,expected_chunks,top5_chunks,expected_rank_in_top5


### Part B — RAG Answer Generation

#### 5. Load FLAN-T5 directly


In [51]:
GENERATION_MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_MODEL_NAME
)

generation_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL_NAME
)

generation_model.eval()

device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

generation_model.to(device)

print("Generation model:", GENERATION_MODEL_NAME)
print("Device:", device)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1132.83it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generation model: google/flan-t5-small
Device: cpu


In [52]:
def build_context(retrieved_df):

    parts = []

    for _, row in retrieved_df.iterrows():

        parts.append(
            f"[{row['chunk_id']}] "
            f"Category: {row['category']}\n"
            f"Section: {row['section']}\n"
            f"Content: {row['content']}"
        )

    return "\n\n".join(parts)


def build_rag_prompt(question, context):

    return f"""You are an RMIT student support assistant.

Answer the student's question using ONLY the RMIT source context.

Rules:
1. Do not use information outside the provided context.
2. Do not invent deadlines, eligibility rules, procedures, or policy requirements.
3. If the context does not provide enough information, say so clearly.
4. Give a concise, direct answer.
5. Preserve important conditions and timeframes.
6. At the end of each factual statement, include the supporting chunk ID in square brackets, for example [SC-01_8].
7. Do not cite a chunk that does not support the statement.

STUDENT QUESTION:
{question}

RMIT SOURCE CONTEXT:
{context}

ANSWER:
"""


def generate_rag_answer(
    question,
    retrieved_df,
    max_input_tokens=1536,
    max_new_tokens=180
):

    context = build_context(
        retrieved_df
    )

    prompt = build_rag_prompt(
        question,
        context
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return (
        answer.strip(),
        context,
        prompt
    )

#### 6. Test one answer before running all 12 questions


In [53]:
test_question = test_df.iloc[8]["prompt"]

retrieved = mmr_then_rerank(
    test_question,
    candidate_k=10,
    mmr_k=5,
    lambda_param=0.70
)

answer, context, prompt = generate_rag_answer(
    test_question,
    retrieved
)

print("QUESTION:")
print(test_question)

print("\nRETRIEVED CHUNKS:")
print(
    retrieved["chunk_id"].tolist()
)

print("\nGENERATED ANSWER:")
print(answer)

QUESTION:
I missed a test because I was unwell, How long do I have to apply for special consideration? I have an EAA deferral thing in my plan.

RETRIEVED CHUNKS:
['EAA-02_43', 'SC-01_8', 'POL-01_63', 'EAA-01_18', 'POL-01_62']

GENERATED ANSWER:
You must register with Equitable Learning Services (ELS).


#### 7. Check whether the generated answer uses valid retrieved chunk IDs

This is not a factual correctness score. It only checks citation formatting/coverage.

A manual review is still required to determine whether the cited chunk actually supports the claim.


In [54]:
def extract_citations(answer):

    return re.findall(
        r"\[([A-Za-z0-9_-]+)\]",
        answer
    )


def citation_diagnostics(
    answer,
    retrieved_chunks
):

    citations = extract_citations(answer)

    valid = [
        c for c in citations
        if c in retrieved_chunks
    ]

    invalid = [
        c for c in citations
        if c not in retrieved_chunks
    ]

    return {
        "citations": citations,
        "valid_citations": valid,
        "invalid_citations": invalid,
        "has_citation": len(citations) > 0,
        "all_citations_valid": (
            len(invalid) == 0
        )
    }


print(
    citation_diagnostics(
        answer,
        retrieved["chunk_id"].tolist()
    )
)

{'citations': [], 'valid_citations': [], 'invalid_citations': [], 'has_citation': False, 'all_citations_valid': True}


#### Part C — Generate answers for all 12 questions


In [55]:
answer_rows = []

for test_id, row in test_df.iterrows():

    question = str(row["prompt"])

    expected_chunks = row["chunks"]

    retrieved = mmr_then_rerank(
        question,
        candidate_k=10,
        mmr_k=5,
        lambda_param=0.70
    )

    retrieved_chunks = (
        retrieved["chunk_id"].tolist()
    )

    generated_answer, context, rag_prompt = (
        generate_rag_answer(
            question,
            retrieved
        )
    )

    citation_info = citation_diagnostics(
        generated_answer,
        retrieved_chunks
    )

    expected_ranks = [
        retrieved_chunks.index(chunk) + 1
        for chunk in expected_chunks
        if chunk in retrieved_chunks
    ]

    expected_answer = None

    if ANSWER_FIELD is not None:
        value = row[ANSWER_FIELD]

        if not pd.isna(value):
            expected_answer = str(value)

    answer_rows.append({

        "test_id": test_id,

        "prompt": question,

        "expected_chunks": expected_chunks,

        "expected_answer": expected_answer,

        "top1_chunk": retrieved_chunks[0],

        "top3_chunks": retrieved_chunks[:3],

        "top5_chunks": retrieved_chunks[:5],

        "expected_rank_in_top5": (
            min(expected_ranks)
            if expected_ranks
            else None
        ),

        "generated_answer": generated_answer,

        "citations": citation_info["citations"],

        "valid_citations": (
            citation_info["valid_citations"]
        ),

        "invalid_citations": (
            citation_info["invalid_citations"]
        ),

        "has_citation": (
            citation_info["has_citation"]
        ),

        "all_citations_valid": (
            citation_info["all_citations_valid"]
        ),

        "retrieved_context": context,

        "rag_prompt": rag_prompt
    })

    print(
        f"Completed {test_id + 1}/{len(test_df)}"
    )

answer_df = pd.DataFrame(
    answer_rows
)

print(
    "Generated answers:",
    len(answer_df)
)

Completed 1/12
Completed 2/12
Completed 3/12
Completed 4/12
Completed 5/12
Completed 6/12
Completed 7/12
Completed 8/12
Completed 9/12
Completed 10/12
Completed 11/12
Completed 12/12
Generated answers: 12


#### 8. View the generated answers


In [56]:
display(
    answer_df[
        [
            "test_id",
            "prompt",
            "top5_chunks",
            "generated_answer",
            "valid_citations",
            "invalid_citations"
        ]
    ]
)

,test_id,prompt,top5_chunks,generated_answer,valid_citations,invalid_citations
0,0,I've got a timed quiz that I won't be able to ...,"[EXT-01_1, EXT-01_0, EAA-02_38, EXT-01_2, POL-...","(40) Where an application is approved, student...",[],[]
1,1,"I've recently lost my job, is this a valid rea...","[SC-01_7, SC-01_13, SC-01_4, SC-01_8, POL-01_63]",You may be eligible to apply for special consi...,[],[]
2,2,So my hard drive with my assignment stopped wo...,"[SC-01_16, SC-01_6, EXT-01_0, EXT-01_1, EAA-02...",You may apply for an extension to the due date...,[],[]
3,3,I've just booked my first appointment with ELS...,"[EAA-02_29, EAA-01_18, EAA-02_28, EAA-02_20, E...",Equitable Learning Service as well as your tea...,[],[]
4,4,I emailed my course coordinator three days ago...,"[EAA-02_34, EAA-02_38, EXT-01_0, POL-01_62, SC...",[EAA-02_34] Category: Equitable Assessment Arr...,[EAA-02_34],[]
5,5,Having a hard time reading through journal art...,"[EAA-02_46, EAA-02_51, SC-01_15, EAA-02_52, EA...",You can find dictation software to convert spe...,[],[]
6,6,My mate got flagged for plagiarism because he ...,"[POL-01_70, EXT-01_3, POL-01_56, EAA-02_19, PO...","(79) The Chief Executive Officer, RMIT UP main...",[],[]
7,7,Who actually sets RMIT's official academic cal...,"[POL-01_67, EAA-02_50, SC-01_11, EAA-02_52, EX...",Depending on your reason for applying for spec...,[],[]
8,8,"I missed a test because I was unwell, How long...","[EAA-02_43, SC-01_8, POL-01_63, EAA-01_18, POL...",You must register with Equitable Learning Serv...,[],[]
9,9,My nan just passed away but I've got a group a...,"[EAA-02_41, EAA-02_34, EXT-01_0, POL-01_62, EX...",You can apply for an extension for a group ass...,[],[]


#### Part D — Manual Answer Evaluation

Because test_prompts.json does not contain gold answers, these fields must be completed by reviewing each generated answer against its retrieved RMIT evidence.

Use:

- correct — factual answer is correct
- complete — addresses the student's question
- grounded — claims are supported by retrieved chunks
- hallucination — unsupported information was added
- comments — brief explanation


In [57]:
manual_review_df = answer_df[
    [
        "test_id",
        "prompt",
        "expected_chunks",
        "top5_chunks",
        "generated_answer",
        "valid_citations"
    ]
].copy()

manual_review_df["correct"] = ""
manual_review_df["complete"] = ""
manual_review_df["grounded"] = ""
manual_review_df["hallucination"] = ""
manual_review_df["comments"] = ""

display(manual_review_df)

,test_id,prompt,expected_chunks,top5_chunks,generated_answer,valid_citations,correct,complete,grounded,hallucination,comments
0,0,I've got a timed quiz that I won't be able to ...,[EXT-01_1],"[EXT-01_1, EXT-01_0, EAA-02_38, EXT-01_2, POL-...","(40) Where an application is approved, student...",[],,,,,
1,1,"I've recently lost my job, is this a valid rea...",[SC-01_7],"[SC-01_7, SC-01_13, SC-01_4, SC-01_8, POL-01_63]",You may be eligible to apply for special consi...,[],,,,,
2,2,So my hard drive with my assignment stopped wo...,[SC-01_16],"[SC-01_16, SC-01_6, EXT-01_0, EXT-01_1, EAA-02...",You may apply for an extension to the due date...,[],,,,,
3,3,I've just booked my first appointment with ELS...,[EAA-02_29],"[EAA-02_29, EAA-01_18, EAA-02_28, EAA-02_20, E...",Equitable Learning Service as well as your tea...,[],,,,,
4,4,I emailed my course coordinator three days ago...,[EAA-02_38],"[EAA-02_34, EAA-02_38, EXT-01_0, POL-01_62, SC...",[EAA-02_34] Category: Equitable Assessment Arr...,[EAA-02_34],,,,,
5,5,Having a hard time reading through journal art...,[EAA-02_46],"[EAA-02_46, EAA-02_51, SC-01_15, EAA-02_52, EA...",You can find dictation software to convert spe...,[],,,,,
6,6,My mate got flagged for plagiarism because he ...,[POL-01_70],"[POL-01_70, EXT-01_3, POL-01_56, EAA-02_19, PO...","(79) The Chief Executive Officer, RMIT UP main...",[],,,,,
7,7,Who actually sets RMIT's official academic cal...,[POL-01_67],"[POL-01_67, EAA-02_50, SC-01_11, EAA-02_52, EX...",Depending on your reason for applying for spec...,[],,,,,
8,8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]","[EAA-02_43, SC-01_8, POL-01_63, EAA-01_18, POL...",You must register with Equitable Learning Serv...,[],,,,,
9,9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]","[EAA-02_41, EAA-02_34, EXT-01_0, POL-01_62, EX...",You can apply for an extension for a group ass...,[],,,,,


In [58]:
citation_summary = pd.DataFrame({
    "Metric": [
        "Answers with a citation",
        "Answers with all citations valid",
        "Answers with invalid citations"
    ],
    "Result": [
        answer_df["has_citation"].mean(),
        answer_df["all_citations_valid"].mean(),
        (
            answer_df["invalid_citations"]
            .apply(len)
            .gt(0)
            .mean()
        )
    ]
})

display(
    citation_summary.style.format(
        {"Result": "{:.1%}"}
    )
)

,Metric,Result
0,Answers with a citation,8.3%
1,Answers with all citations valid,100.0%
2,Answers with invalid citations,0.0%


In [59]:
print("RETRIEVAL SUMMARY")

display(
    retrieval_summary.style.format(
        {"Result": "{:.1%}"}
    )
)

print("\nGENERATION/CITATION SUMMARY")

display(
    citation_summary.style.format(
        {"Result": "{:.1%}"}
    )
)

print(
    "\nManual answer evaluation is required for factual correctness, "
    "completeness, grounding and hallucination."
)

RETRIEVAL SUMMARY


,Metric,Result
0,Any expected — Top-1,66.7%
1,Any expected — Top-3,83.3%
2,Any expected — Top-5,100.0%
3,All expected — Top-1,58.3%
4,All expected — Top-3,75.0%
5,All expected — Top-5,83.3%



GENERATION/CITATION SUMMARY


,Metric,Result
0,Answers with a citation,8.3%
1,Answers with all citations valid,100.0%
2,Answers with invalid citations,0.0%



Manual answer evaluation is required for factual correctness, completeness, grounding and hallucination.


In [60]:
display(
    answer_df[["prompt", "generated_answer"]].style.set_properties(
        **{"white-space": "pre-wrap", "text-align": "left"}
    )
)

,prompt,generated_answer
0,"I've got a timed quiz that I won't be able to complete on time, am I able to get an extension?","(40) Where an application is approved, students meet the new assessment due date for submission of work or to undertake the assessment where attendance is required."
1,"I've recently lost my job, is this a valid reason to apply for special consideration?",You may be eligible to apply for special consideration.
2,"So my hard drive with my assignment stopped working and I didn't backup my work, will I be able to get an extension or special consideration?",You may apply for an extension to the due date of up to seven calendar days
3,I've just booked my first appointment with ELS next week and I'm feeling nervous. Is there anything I need to bring or have ready before I go in?,Equitable Learning Service as well as your teachers and tutors and perhaps some additional supports such as the Careers team or the Library study supports or mental health supports such as your counsellor and that might be RMIT's counselling service or your own personal counsellor or psychologist and other medical supports who might be able to assist through during times of stress or just give you some strategies on how to cope up during the studies at RMIT.
4,I emailed my course coordinator three days ago asking for an extension and I've heard absolutely nothing back and the due date is tomorrow — what do I do now?,[EAA-02_34] Category: Equitable Assessment Arrangements Section: How do I ask for an extension?
5,"Having a hard time reading through journal articles for an assessment as I'm dyslexic, is there some kind of text-to-speech thing RMIT can set me up with?",You can find dictation software to convert speech to text.
6,"My mate got flagged for plagiarism because he reused part of an assignment he wrote for a different course last year, does RMIT actually define that as plagiarism?","(79) The Chief Executive Officer, RMIT UP maintains the assessment processes for RMIT UP."
7,Who actually sets RMIT's official academic calendar each year? I'm trying to plan around census dates.,"Depending on your reason for applying for special consideration, you may submit: an Impact Assessment Statement (PDF, 45KB) completed by your treating health practitioner."
8,"I missed a test because I was unwell, How long do I have to apply for special consideration? I have an EAA deferral thing in my plan.",You must register with Equitable Learning Services (ELS).
9,"My nan just passed away but I've got a group assignment due friday, can I even apply for an extension on something like this? How much extra time could I get?",You can apply for an extension for a group assessment with a due date?


In [ ]:
test_question = "What is the process for applying for a scholarship at RMIT University?"

retrieved = mmr_then_rerank(
    test_question,
    candidate_k=10,
    mmr_k=5,
    lambda_param=0.70
)

answer, context, prompt = generate_rag_answer(
    test_question,
    retrieved
)

print("QUESTION:")
print(test_question)

print("\nRETRIEVED CHUNKS:")
print(
    retrieved["chunk_id"].tolist()
)

print("\nGENERATED ANSWER:")
print(answer)

QUESTION:
What is the process for applying for a scholarship at RMIT University?

RETRIEVED CHUNKS:
['POL-01_67', 'EXT-01_2', 'EAA-02_51', 'SC-01_9', 'SC-01_15']

GENERATED ANSWER:
You must apply at least one working day before the assessment deadline. If you have access to Canvas, use the Assessment Extensions Tool. For help using the tool, refer to the Student Assessment Extensions Tool Guide (PDF, 334 KB) . If you don’t respond to a request for more information or documents, we may have to cancel your application. You’ll receive an outcome email up to 10 working days after the notification that your application is complete. The email will state the outcome for each assessment task listed in your application. The standard outcomes are ‘granted’, ‘denied’, ‘cancelled’ and ‘withdrawn’ If you’re granted special consideration, it’s important that you follow any instructions and recommendations in your outcome email. Where an outcome requires you to contact the course coordinator by a giv

: 